# 建模（经过特征工程）

前面baseline已经跑通了，接下来可以施展feature set B, set C了。先接入一下数据，然后想办法把前面的feature Engineering固定成函数

In [2]:
from pathlib import Path

import pandas as pd
from sklearn.base import BaseEstimator

CURRENT_DIR = Path().cwd()
if CURRENT_DIR.name == "notebooks":
    #当前在notebooks/下面
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    #当前在根目录
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

df = pd.read_csv(PROCESSED_DIR / "train_cleaned.csv")
df["MSSubClass"] = df["MSSubClass"].astype(str)

然后把X和y拿出来

In [3]:
X = df.drop(columns="SalePrice")
y = df["SalePrice"]

接下来开始写特征工程的部分。Set A就是什么也不做，直接完成后面的preprocessing，然后未给模型。而Set B就是先统计模型选择，再preprocessing；Set C就是先手工，再统计选择，再preprocessing。

预处理直接利用现成工具组装，不用之前自己写的了。说白了就是一个编码和一个转换器，都有现成工具

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import RobustScaler
import numpy as np
# 定义数据集
num_cols = X.select_dtypes("number").columns
cat_cols = X.select_dtypes("str").columns


In [150]:
# # 这一段可注释，不能删，这是手动实现的preprocessing
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.preprocessing import RobustScaler
#
# def preprocessing(X_train, X_val, y_train, y_val):
#     # Scaling and 1-hot Encoding
#     #实例化一个encoder
#     encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
#
#     # 首先获取所有数据集的所有的类别变量列及其列名
#     # 先把训练集所有的类别变量列取出来
#     cat_columns = X_train.select_dtypes("str")
#     cat_columns_name = cat_columns.columns.tolist()
#
#     # 再把验证集所有的类别变量列取出来
#     cat_columns_Xval = X_val[cat_columns_name]
#
#     # 编码
#     encoded_cat_Xtrain = encoder.fit_transform(cat_columns)
#     encoded_cat_Xval = encoder.transform(cat_columns_Xval)
#
#     # 把编码后的列表转化成Dataframe
#     encoded_cat_Xtrain_df = pd.DataFrame(
#         encoded_cat_Xtrain,
#         columns=encoder.get_feature_names_out(cat_columns_name),
#         index=X_train.index
#     )
#
#     encoded_cat_Xval_df = pd.DataFrame(
#         encoded_cat_Xval,
#         columns=encoder.get_feature_names_out(cat_columns_name),
#         index=X_val.index
#     )
#
#     # 把验证集和测试集各自拼接到一起
#     X_train = pd.concat([X_train.drop(columns=cat_columns_name), encoded_cat_Xtrain_df], axis=1)
#     X_val = pd.concat([X_val.drop(columns=cat_columns_name), encoded_cat_Xval_df], axis=1)
#
#     # 对数处理y
#     y_train = np.log(y_train)
#     y_val = np.log(y_val)
#
#     #
#     # 新建一个转换器对象
#     scaler = RobustScaler()
#     # 取出数值列
#     num_features = X_train.select_dtypes("number").columns.tolist()
#
#     # 更新数值列
#     X_train[num_features] = scaler.fit_transform(X_train[num_features])
#     X_val[num_features] = scaler.transform(X_val[num_features])
#
#     return X_train, X_val, y_train, y_val

接下来是定义三种不同的Feature set，我们需要把两组处理方法写成函数，手工处理的部分非常简单，直接移到函数里。

In [5]:
def manual_feature_engineering(X_input):
    # * 年份可以用差值来代替建造时间
    X_input["GarageAge"] = 0
    mask = X_input["HasGarage"] == 1
    X_input.loc[mask, "GarageAge"] = (X_input.loc[mask, "YrSold"] - X_input.loc[mask, "GarageYrBlt"])
    X_input["HouseAge"] = X_input["YrSold"] - X_input["YearBuilt"]
    X_input["RemodAge"] = X_input["YrSold"] - X_input["YearRemodAdd"]

    # * 可以把所有的面积加起来计算总使用面积
    X_input["TotalUtilSF"] = X_input["1stFlrSF"] + X_input["2ndFlrSF"] + X_input["TotalBsmtSF"]

    # * 卫生间等效加总
    X_input["EqvlBath"] = X_input["BsmtFullBath"] + 0.5 * X_input["BsmtHalfBath"] + X_input["FullBath"] + 0.5 * X_input["HalfBath"]

    # * 比例系列
    X_input["LivingRatio"] = X_input["GrLivArea"] / X_input["LotArea"]
    X_input["BedroomRatio"] = X_input["Bedroom"] / X_input["TotalUtilSF"]

    # * 面积×质量
    X_input["OvrallMul"] = X_input["OverallQual"] * X_input["LotArea"]
    X_input["BsmtMul"] = X_input["BsmtQual"] * X_input["TotalBsmtSF"]
    X_input["KitMul"] = X_input["Kitchen"] * X_input["KitchenQual"] # 这个是数量 * 质量

    # * Porch 总面积
    X_input["TotalPorchSF"] = X_input["OpenPorchSF"] + X_input["EnclosedPorch"] + X_input["3SsnPorch"] + X_input["ScreenPorch"]

    # * 设施存在性
    # HasGarage已经有了
    X_input["HasPool"] = 1 * (X_input["PoolArea"] > 0)
    X_input["HasBsmt"] = 1 * (X_input["BsmtQual"] > 0)
    X_input["HasFireplace"] = 1 * (X_input["Fireplaces"] > 0)
    X_input["Has2ndfloor"] = 1 * (X_input["2ndFlrSF"] > 0)
    X_input["HasPorch"] = 1 * (X_input["TotalPorchSF"] > 0)
    X_input["HasFence"] = 1 * (X_input["Fence"] > 0)

    # * 总设施数量
    X_input["Totalutil"] = X_input["HasPool"] + X_input["HasGarage"] + X_input["HasBsmt"] + X_input["HasFireplace"] + X_input["Has2ndfloor"] + X_input["HasPorch"] + X_input["HasFence"]

    # * 根号项
    X_input["RootUtil"] = X_input["Totalutil"] ** 0.5
    X_input["RootLotArea"] = X_input["LotArea"] ** 0.5

    return X_input

特征工程完成以后，还有特征selection，我这里用BIC为准则，自动选择合适的数量。利用backward计数。也就是

* Set A: 数据→预处理→fit
* Set B: 数据→预处理→BIC→fit
* Set C: 数据→手工特征工程→预处理→BIC→fit

接下来我们先解决BIC的问题，由于暂时没有非常方便的，可以兼容sklearn的函数可以直接调用，我们需要手写。大概包含三步问题：
* 写BIC的计算公式
* 写基于BIC的选择方法
* 把BIC函数包装成适配transformer的形式

首先，我们需要写一个计算函数，这一步比较简单：
* residual = y - y_pred
* RSS = sum(residual²)
* BIC = n * log(RSS / n) + k * log(n)

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

def calculate_BIC(X_train, y_train):
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_train)
    MSE = mean_squared_error(y_train, y_pred)
    nrows = X_train.shape[0]
    ncols = X_train.shape[1]
    RSS = MSE * nrows
    return nrows * np.log(RSS / nrows) + (ncols + 1) * np.log(nrows)

完成计算BIC的公式，接下来要写一个函数，根据BIC自动选择应该保留的列。我们采用backward选择，让模型不断计算，最终得到一个删除任何列都会导致BIC分数更高的程度，这个时候就算选择好了。

In [7]:
import time

def BIC_selector(X_train, y_train):
    # 先获得初始BIC
    current_BIC = calculate_BIC(X_train, y_train)
    # 初始列名
    all_cols_name = X_train.columns.tolist()

    X_train_copy = X_train.copy()
    # 准备变量
    cols_to_drop = ""
    history_bic = []
    step = 1
    # 进行循环，直到最后选出所有的列名
    print("进入循环.....")
    start_time = time.time()
    while True:
        best_BIC = current_BIC
        # 设置flag，如果一轮中发现了更小的可删除列，就为真，否则为假
        smaller_bic_founded = False
        # 在当前的所有列里面循环
        for cols in all_cols_name:
            # 删除一个列之后的BIC
            new_BIC = calculate_BIC(X_train_copy.drop(cols, axis=1), y_train)
            # 如果刚刚删除一个列得到的BIC更小，就记下来，并更新当前最小的BIC
            if new_BIC < best_BIC:
                cols_to_drop = cols
                best_BIC = new_BIC
                smaller_bic_founded = True
        # 如果一个循环结束，没有发现任何新的可删除列
        # 就说明已经穷尽了，直接跳出while
        if not smaller_bic_founded:
            break
        # 如果只剩下1列，也直接跳出
        if X_train_copy.shape[1] <= 1:
            break
        # 更新阶段，把该删的删掉，更新“当前所有列”，记录历史
        end_time = time.time()
        print(f"已经完成一次删列，当前已删除列数: {step}, 当前耗时 {int(end_time - start_time)} 秒.")
        X_train_copy = X_train_copy.drop(cols_to_drop, axis=1)
        current_BIC = best_BIC
        all_cols_name = X_train_copy.columns.tolist()
        history_bic.append((step, current_BIC))
        step +=1

    print("完成变量选择.")
    # 返回处理后的子数据集
    return X_train_copy



代码写好了，我们计算一下试试，用之前的数据集看看效果。我们先写一个测试单元，先把数据准备好，后面可能会多次用到这个测试单元。

In [8]:
# 获得数据
X_preprocessed = X.copy()
y_preprocessed = y.copy()


# # 先完成预处理
# num_cols_name = X_preprocessed.select_dtypes("number").columns.tolist()
# cat_cols_name = X_preprocessed.select_dtypes("str").columns.tolist()
# # 处理类别列
# one_hot_encoder = OneHotEncoder(handle_unknown='ignore',
#                                 sparse_output=False,
#                                 drop="first")
# encoded_cat_cols = one_hot_encoder.fit_transform(X_preprocessed[cat_cols_name])
# encoded_cat_cols = pd.DataFrame(
#     encoded_cat_cols,
#     columns = one_hot_encoder.get_feature_names_out(cat_cols_name),
#     index = X_preprocessed.index
# )
# # 处理数值列
# robustscaler = RobustScaler()
# X_preprocessed[num_cols_name] = robustscaler.fit_transform(X_preprocessed[num_cols_name])
# # 拼接
# X_preprocessed = pd.concat([X_preprocessed[num_cols_name], encoded_cat_cols], axis=1)
# # 处理y
# y_preprocessed = np.log(y_preprocessed)

数据准备好以后，就开始测试这个函数

In [155]:
# # 测试，跑一次13分钟，不需要时候注释掉
# selected_X = BIC_selector(X_preprocessed,y_preprocessed)
# selected_X.shape

没问题，就是跑一次时间相当长，因为对于200多个列，需要对200多个列依次测试并计算分数，测试每个列的时候都要完整的拟合一次，最后删除了那么多列，相当于200*200，总共要拟合3 - 4万次，所以计算量相当大，跑完一次大概13分钟。

我们接下来要把这个代码变成符合transformer的形式，要把fit和transform分开

In [9]:
from sklearn.base import TransformerMixin
class BICSelector(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    def fit(self, X_train, y_train):

        # 这里运行backward BIC
        # 先获得初始BIC
        current_BIC = calculate_BIC(X_train, y_train)
        # 初始列名，维护一个当前存在的列
        all_cols_name = X_train.columns.tolist()
        # selected_columns = X_train.columns.tolist()

        X_train_copy = X_train.copy()
        # 准备变量
        cols_to_drop = None
        history_bic = []
        step = 1
        # 进行循环，直到最后选出所有的列名
        print("进入循环，开始测试各列并计算分数")
        start_time = time.time()
        while True:
            # 首先如果只剩一列，直接跳出
            if X_train_copy.shape[1] <= 1:
                break
            #
            best_BIC = current_BIC

            # 不需要flag，改成cols_to_drop
            cols_to_drop = None
            # 在当前的所有列里面循环
            for cols in all_cols_name:
                # 逐个删除列，逐个计算BIC
                new_BIC = calculate_BIC(X_train_copy.drop(cols, axis=1), y_train)
                # 如果刚刚删除一个列得到的BIC更小，就记下来，并更新当前最小的BIC
                if new_BIC < best_BIC:
                    cols_to_drop = cols
                    best_BIC = new_BIC
            # 如果一个循环结束，没有发现任何新的可删除列
            # 就说明已经穷尽了，直接跳出while
            if cols_to_drop is None:
                break

            # 更新阶段，把该删的删掉，更新“当前所有列”，记录历史
            # 删除本轮确定的列，更新当前最低BIC，更新当前列名
            end_time = time.time()
            print(f"已经完成一次删列，当前已删除列数: {step}, 当前BIC: {round(best_BIC, 2)}, 当前耗时 {int(end_time - start_time)} 秒.")
            X_train_copy = X_train_copy.drop(cols_to_drop, axis=1)
            current_BIC = best_BIC
            all_cols_name = X_train_copy.columns.tolist()
            history_bic.append((step, cols_to_drop, current_BIC))
            step +=1

        self.selected_columns_ = X_train_copy.columns.tolist()
        self.final_bic_ = current_BIC
        self.history_bic_ = history_bic

        return self
    def transform(self, X_train):
        return X_train[self.selected_columns_]

写完Class，我们先看看这个BICSelector能不能正常使用

In [157]:
# 不要重复跑这个代码块，以免覆盖结果或重新跑
selector = BICSelector()
selector.fit(X_preprocessed, y_preprocessed)
X_selected = selector.transform(X_preprocessed)

print("Before:", X_preprocessed.shape)
print("After:", X_selected.shape)

print("Final BIC:", selector.final_bic_)
print("Selected columns:", selector.selected_columns_)

进入循环，开始测试各列并计算分数
已经完成一次删列，当前已删除列数: 1, 当前BIC: -5192.27, 当前耗时 11 秒.
已经完成一次删列，当前已删除列数: 2, 当前BIC: -5199.55, 当前耗时 19 秒.
已经完成一次删列，当前已删除列数: 3, 当前BIC: -5206.83, 当前耗时 27 秒.
已经完成一次删列，当前已删除列数: 4, 当前BIC: -5214.11, 当前耗时 35 秒.
已经完成一次删列，当前已删除列数: 5, 当前BIC: -5221.39, 当前耗时 43 秒.
已经完成一次删列，当前已删除列数: 6, 当前BIC: -5228.67, 当前耗时 51 秒.
已经完成一次删列，当前已删除列数: 7, 当前BIC: -5235.95, 当前耗时 59 秒.
已经完成一次删列，当前已删除列数: 8, 当前BIC: -5243.23, 当前耗时 76 秒.
已经完成一次删列，当前已删除列数: 9, 当前BIC: -5250.51, 当前耗时 87 秒.
已经完成一次删列，当前已删除列数: 10, 当前BIC: -5257.79, 当前耗时 97 秒.
已经完成一次删列，当前已删除列数: 11, 当前BIC: -5265.07, 当前耗时 105 秒.
已经完成一次删列，当前已删除列数: 12, 当前BIC: -5272.35, 当前耗时 118 秒.
已经完成一次删列，当前已删除列数: 13, 当前BIC: -5279.63, 当前耗时 127 秒.
已经完成一次删列，当前已删除列数: 14, 当前BIC: -5286.9, 当前耗时 144 秒.
已经完成一次删列，当前已删除列数: 15, 当前BIC: -5294.18, 当前耗时 174 秒.
已经完成一次删列，当前已删除列数: 16, 当前BIC: -5301.46, 当前耗时 181 秒.
已经完成一次删列，当前已删除列数: 17, 当前BIC: -5308.73, 当前耗时 192 秒.
已经完成一次删列，当前已删除列数: 18, 当前BIC: -5315.99, 当前耗时 201 秒.
已经完成一次删列，当前已删除列数: 19, 当前BIC: -5323.26, 当前耗时 209 秒.
已经完成一次删列，当前已删除列数: 20, 当前BIC: -5330

前期基本上每删除一列，就会下降7左右的BIC，这就说明RSS基本没怎么变，也就是说，这个时候删除的都是对模型没什么效果的冗余变量。最后跑了14分钟，将近15分钟，删列删到还剩54列，这里也可以看他删列的顺序。

最先删除的是一些杂项，有趣的是没想到TotalBsmntSF很早删除了，我个人猜测可能是因为这个量和建筑总面积共线了，因此被删了，而HasGarage也删除了，可能也是共线的原因，Exterior系列删的很多，这个词条的解释是，外部cover的材料，可能大家交易的时候，都不太在意这个因素吧

In [158]:
# 不要重复跑这个代码块，以免覆盖结果或重新跑
selector.history_bic_

[(1, 'MSSubClass_90', np.float64(-5192.272902850062)),
 (2, 'TotalBsmtSF', np.float64(-5199.552911102946)),
 (3, '1stFlrSF', np.float64(-5206.832919355829)),
 (4, 'HasGarage', np.float64(-5214.112927608713)),
 (5, 'Exterior1st_CBlock', np.float64(-5221.392935861598)),
 (6, 'Neighborhood_BrDale', np.float64(-5228.67294058377)),
 (7, 'MSSubClass_80', np.float64(-5235.952935226016)),
 (8, 'LandContour_Low', np.float64(-5243.232886279272)),
 (9, 'Condition1_RRNe', np.float64(-5250.512584901298)),
 (10, 'Neighborhood_NPkVill', np.float64(-5257.792154730395)),
 (11, 'Neighborhood_SawyerW', np.float64(-5265.071158626004)),
 (12, 'RoofStyle_Mansard', np.float64(-5272.349960136897)),
 (13, 'Exterior2nd_BrkFace', np.float64(-5279.627924312515)),
 (14, 'Heating_Grav', np.float64(-5286.904154253581)),
 (15, 'MSSubClass_190', np.float64(-5294.180328518559)),
 (16, 'ExterQual', np.float64(-5301.455979907742)),
 (17, 'Neighborhood_SWISU', np.float64(-5308.726953553665)),
 (18, 'Exterior1st_VinylSd', 

接下来组装，大概目标流程是：
```
原始 X
      ↓
ColumnTransformer
      ↓
RobustScaler + OneHot
      ↓
BICSelector
      ↓
LinearRegression
```
这其中的问题是，BICSelector全程用的是pandas，而ColumnTransformer用的是ndarray，不是DataFrame格式

In [10]:
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore",
                              drop="first",
                              sparse_output=False),
         cat_cols),
    ]
)
preprocessor.set_output(transform="pandas")

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("selector", BICSelector()),
        ("LRmodel", LinearRegression()),
    ]
)

# 测试pipeline
# 数据应该是刚刚从df里面分出来的X和y，y提前取log就行了，其他的不要做，不能重复处理
pipeline.fit(X_preprocessed, np.log(y_preprocessed))
print("Pipeline fitting completed.") # 加一个防止输出html进而导致渲染错误

进入循环，开始测试各列并计算分数
已经完成一次删列，当前已删除列数: 1, 当前BIC: -5192.27, 当前耗时 9 秒.
已经完成一次删列，当前已删除列数: 2, 当前BIC: -5199.55, 当前耗时 18 秒.
已经完成一次删列，当前已删除列数: 3, 当前BIC: -5206.83, 当前耗时 26 秒.
已经完成一次删列，当前已删除列数: 4, 当前BIC: -5214.11, 当前耗时 34 秒.
已经完成一次删列，当前已删除列数: 5, 当前BIC: -5221.39, 当前耗时 42 秒.
已经完成一次删列，当前已删除列数: 6, 当前BIC: -5228.67, 当前耗时 50 秒.
已经完成一次删列，当前已删除列数: 7, 当前BIC: -5235.95, 当前耗时 58 秒.
已经完成一次删列，当前已删除列数: 8, 当前BIC: -5243.23, 当前耗时 65 秒.
已经完成一次删列，当前已删除列数: 9, 当前BIC: -5250.51, 当前耗时 72 秒.
已经完成一次删列，当前已删除列数: 10, 当前BIC: -5257.79, 当前耗时 80 秒.
已经完成一次删列，当前已删除列数: 11, 当前BIC: -5265.07, 当前耗时 87 秒.
已经完成一次删列，当前已删除列数: 12, 当前BIC: -5272.35, 当前耗时 103 秒.
已经完成一次删列，当前已删除列数: 13, 当前BIC: -5279.63, 当前耗时 110 秒.
已经完成一次删列，当前已删除列数: 14, 当前BIC: -5286.9, 当前耗时 117 秒.
已经完成一次删列，当前已删除列数: 15, 当前BIC: -5294.18, 当前耗时 125 秒.
已经完成一次删列，当前已删除列数: 16, 当前BIC: -5301.46, 当前耗时 131 秒.
已经完成一次删列，当前已删除列数: 17, 当前BIC: -5308.73, 当前耗时 138 秒.
已经完成一次删列，当前已删除列数: 18, 当前BIC: -5315.99, 当前耗时 144 秒.
已经完成一次删列，当前已删除列数: 19, 当前BIC: -5323.26, 当前耗时 151 秒.
已经完成一次删列，当前已删除列数: 20, 当前BIC: -5330.5

好，pipeline没问题，我们应该可以正常的输出训练内容

In [ ]:
selector = pipeline.named_steps["selector"]

print("最终保留特征数:", len(selector.selected_columns_))
print("最终 BIC:", selector.final_bic_)
print("保留的特征:")
print(selector.selected_columns_)

transformer代码没问题，就开始测试交叉测试。注意，这里是Set B的交叉验证。执行一次交叉验证可能需要1个小时左右. 我们为了干净，完整的复制前面的代码，让整个部分的代码都出现在的这个单元格里面。

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
# 定义数据集
num_cols = X.select_dtypes("number").columns
cat_cols = X.select_dtypes("str").columns

print("数据已经准备好.")
# 定义preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore",
                              drop="first",
                              sparse_output=False),
         cat_cols),
    ]
)
preprocessor.set_output(transform="pandas")
print("处理器已定义.")

# 把处理器和模型连接起来
# 数据管道由两方面构成，一个是数据处理器，一个是线性模型，SetB没有特征工程环节。
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("selector", BICSelector()),
        ("LRmodel", LinearRegression()),
    ]
)
print("数据管道与模型已组装.")


# 做cross validate
# 定义Fold
cv = KFold(n_splits=5, shuffle=True, random_state=42)
# 使用cross_validate计算
# 先定义scorings，是我们需要用到的评价指标
scorings={
        "RMSE": "neg_root_mean_squared_error",
        "MAE": "neg_mean_absolute_error",
        "MSE": "neg_mean_squared_error",
        "R2": "r2"
    }

scores = cross_validate(
    pipeline,
    X,
    np.log(y),
    cv=cv,
    scoring=scorings,
    return_train_score=True
)
print("交叉验证已完成.")

数据已经准备好.
处理器已定义.
数据管道与模型已组装.
进入循环，开始测试各列并计算分数
已经完成一次删列，当前已删除列数: 1, 当前BIC: -3943.59, 当前耗时 12 秒.
已经完成一次删列，当前已删除列数: 2, 当前BIC: -3950.65, 当前耗时 31 秒.
已经完成一次删列，当前已删除列数: 3, 当前BIC: -3957.7, 当前耗时 40 秒.
已经完成一次删列，当前已删除列数: 4, 当前BIC: -3964.76, 当前耗时 50 秒.
已经完成一次删列，当前已删除列数: 5, 当前BIC: -3971.82, 当前耗时 59 秒.
已经完成一次删列，当前已删除列数: 6, 当前BIC: -3978.87, 当前耗时 79 秒.
已经完成一次删列，当前已删除列数: 7, 当前BIC: -3985.93, 当前耗时 98 秒.
已经完成一次删列，当前已删除列数: 8, 当前BIC: -3992.98, 当前耗时 105 秒.
已经完成一次删列，当前已删除列数: 9, 当前BIC: -4000.04, 当前耗时 112 秒.
已经完成一次删列，当前已删除列数: 10, 当前BIC: -4007.09, 当前耗时 118 秒.
已经完成一次删列，当前已删除列数: 11, 当前BIC: -4014.14, 当前耗时 125 秒.
已经完成一次删列，当前已删除列数: 12, 当前BIC: -4021.19, 当前耗时 131 秒.
已经完成一次删列，当前已删除列数: 13, 当前BIC: -4028.23, 当前耗时 137 秒.
已经完成一次删列，当前已删除列数: 14, 当前BIC: -4035.27, 当前耗时 143 秒.
已经完成一次删列，当前已删除列数: 15, 当前BIC: -4042.31, 当前耗时 149 秒.
已经完成一次删列，当前已删除列数: 16, 当前BIC: -4049.34, 当前耗时 155 秒.
已经完成一次删列，当前已删除列数: 17, 当前BIC: -4056.37, 当前耗时 161 秒.
已经完成一次删列，当前已删除列数: 18, 当前BIC: -4063.4, 当前耗时 167 秒.
已经完成一次删列，当前已删除列数: 19, 当前BIC: -4070.43, 当前耗时 172 秒.
已经完

E:\Software\PythonProjects\Housing_Prices\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [12, 13] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


进入循环，开始测试各列并计算分数
已经完成一次删列，当前已删除列数: 1, 当前BIC: -4138.84, 当前耗时 13 秒.
已经完成一次删列，当前已删除列数: 2, 当前BIC: -4145.9, 当前耗时 26 秒.
已经完成一次删列，当前已删除列数: 3, 当前BIC: -4152.96, 当前耗时 34 秒.
已经完成一次删列，当前已删除列数: 4, 当前BIC: -4160.01, 当前耗时 41 秒.
已经完成一次删列，当前已删除列数: 5, 当前BIC: -4167.07, 当前耗时 49 秒.
已经完成一次删列，当前已删除列数: 6, 当前BIC: -4174.13, 当前耗时 58 秒.
已经完成一次删列，当前已删除列数: 7, 当前BIC: -4181.18, 当前耗时 66 秒.
已经完成一次删列，当前已删除列数: 8, 当前BIC: -4188.24, 当前耗时 73 秒.
已经完成一次删列，当前已删除列数: 9, 当前BIC: -4195.3, 当前耗时 80 秒.
已经完成一次删列，当前已删除列数: 10, 当前BIC: -4202.35, 当前耗时 86 秒.
已经完成一次删列，当前已删除列数: 11, 当前BIC: -4209.41, 当前耗时 93 秒.
已经完成一次删列，当前已删除列数: 12, 当前BIC: -4216.46, 当前耗时 99 秒.
已经完成一次删列，当前已删除列数: 13, 当前BIC: -4223.52, 当前耗时 105 秒.
已经完成一次删列，当前已删除列数: 14, 当前BIC: -4230.57, 当前耗时 111 秒.
已经完成一次删列，当前已删除列数: 15, 当前BIC: -4237.63, 当前耗时 117 秒.
已经完成一次删列，当前已删除列数: 16, 当前BIC: -4244.68, 当前耗时 130 秒.
已经完成一次删列，当前已删除列数: 17, 当前BIC: -4251.73, 当前耗时 140 秒.
已经完成一次删列，当前已删除列数: 18, 当前BIC: -4258.78, 当前耗时 152 秒.
已经完成一次删列，当前已删除列数: 19, 当前BIC: -4265.83, 当前耗时 161 秒.
已经完成一次删列，当前已删除列数: 20, 当前BIC: -4272.88

E:\Software\PythonProjects\Housing_Prices\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [8, 12, 13, 16, 21] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


进入循环，开始测试各列并计算分数
已经完成一次删列，当前已删除列数: 1, 当前BIC: -4027.52, 当前耗时 7 秒.
已经完成一次删列，当前已删除列数: 2, 当前BIC: -4034.57, 当前耗时 13 秒.
已经完成一次删列，当前已删除列数: 3, 当前BIC: -4041.63, 当前耗时 20 秒.
已经完成一次删列，当前已删除列数: 4, 当前BIC: -4048.69, 当前耗时 27 秒.
已经完成一次删列，当前已删除列数: 5, 当前BIC: -4055.74, 当前耗时 34 秒.
已经完成一次删列，当前已删除列数: 6, 当前BIC: -4062.8, 当前耗时 41 秒.
已经完成一次删列，当前已删除列数: 7, 当前BIC: -4069.86, 当前耗时 53 秒.
已经完成一次删列，当前已删除列数: 8, 当前BIC: -4076.92, 当前耗时 59 秒.
已经完成一次删列，当前已删除列数: 9, 当前BIC: -4083.97, 当前耗时 65 秒.
已经完成一次删列，当前已删除列数: 10, 当前BIC: -4091.03, 当前耗时 72 秒.
已经完成一次删列，当前已删除列数: 11, 当前BIC: -4098.08, 当前耗时 78 秒.
已经完成一次删列，当前已删除列数: 12, 当前BIC: -4105.14, 当前耗时 84 秒.
已经完成一次删列，当前已删除列数: 13, 当前BIC: -4112.19, 当前耗时 90 秒.
已经完成一次删列，当前已删除列数: 14, 当前BIC: -4119.24, 当前耗时 96 秒.
已经完成一次删列，当前已删除列数: 15, 当前BIC: -4126.3, 当前耗时 102 秒.
已经完成一次删列，当前已删除列数: 16, 当前BIC: -4133.35, 当前耗时 108 秒.
已经完成一次删列，当前已删除列数: 17, 当前BIC: -4140.4, 当前耗时 113 秒.
已经完成一次删列，当前已删除列数: 18, 当前BIC: -4147.44, 当前耗时 119 秒.
已经完成一次删列，当前已删除列数: 19, 当前BIC: -4154.49, 当前耗时 126 秒.
已经完成一次删列，当前已删除列数: 20, 当前BIC: -4161.53, 当前

E:\Software\PythonProjects\Housing_Prices\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [8, 12, 14] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


进入循环，开始测试各列并计算分数
已经完成一次删列，当前已删除列数: 1, 当前BIC: -3978.19, 当前耗时 7 秒.
已经完成一次删列，当前已删除列数: 2, 当前BIC: -3985.25, 当前耗时 15 秒.
已经完成一次删列，当前已删除列数: 3, 当前BIC: -3992.31, 当前耗时 22 秒.
已经完成一次删列，当前已删除列数: 4, 当前BIC: -3999.36, 当前耗时 29 秒.
已经完成一次删列，当前已删除列数: 5, 当前BIC: -4006.42, 当前耗时 35 秒.
已经完成一次删列，当前已删除列数: 6, 当前BIC: -4013.48, 当前耗时 42 秒.
已经完成一次删列，当前已删除列数: 7, 当前BIC: -4020.53, 当前耗时 49 秒.
已经完成一次删列，当前已删除列数: 8, 当前BIC: -4027.58, 当前耗时 55 秒.
已经完成一次删列，当前已删除列数: 9, 当前BIC: -4034.64, 当前耗时 62 秒.
已经完成一次删列，当前已删除列数: 10, 当前BIC: -4041.69, 当前耗时 69 秒.
已经完成一次删列，当前已删除列数: 11, 当前BIC: -4048.74, 当前耗时 75 秒.
已经完成一次删列，当前已删除列数: 12, 当前BIC: -4055.79, 当前耗时 82 秒.
已经完成一次删列，当前已删除列数: 13, 当前BIC: -4062.83, 当前耗时 90 秒.
已经完成一次删列，当前已删除列数: 14, 当前BIC: -4069.88, 当前耗时 101 秒.
已经完成一次删列，当前已删除列数: 15, 当前BIC: -4076.93, 当前耗时 107 秒.
已经完成一次删列，当前已删除列数: 16, 当前BIC: -4083.97, 当前耗时 113 秒.
已经完成一次删列，当前已删除列数: 17, 当前BIC: -4091.0, 当前耗时 119 秒.
已经完成一次删列，当前已删除列数: 18, 当前BIC: -4098.03, 当前耗时 125 秒.
已经完成一次删列，当前已删除列数: 19, 当前BIC: -4105.07, 当前耗时 130 秒.
已经完成一次删列，当前已删除列数: 20, 当前BIC: -4112.1, 

E:\Software\PythonProjects\Housing_Prices\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [13, 14] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


进入循环，开始测试各列并计算分数
已经完成一次删列，当前已删除列数: 1, 当前BIC: -3929.84, 当前耗时 7 秒.
已经完成一次删列，当前已删除列数: 2, 当前BIC: -3936.9, 当前耗时 18 秒.
已经完成一次删列，当前已删除列数: 3, 当前BIC: -3943.96, 当前耗时 28 秒.
已经完成一次删列，当前已删除列数: 4, 当前BIC: -3951.02, 当前耗时 40 秒.
已经完成一次删列，当前已删除列数: 5, 当前BIC: -3958.07, 当前耗时 48 秒.
已经完成一次删列，当前已删除列数: 6, 当前BIC: -3965.13, 当前耗时 58 秒.
已经完成一次删列，当前已删除列数: 7, 当前BIC: -3972.19, 当前耗时 65 秒.
已经完成一次删列，当前已删除列数: 8, 当前BIC: -3979.24, 当前耗时 73 秒.
已经完成一次删列，当前已删除列数: 9, 当前BIC: -3986.3, 当前耗时 82 秒.
已经完成一次删列，当前已删除列数: 10, 当前BIC: -3993.35, 当前耗时 92 秒.
已经完成一次删列，当前已删除列数: 11, 当前BIC: -4000.41, 当前耗时 99 秒.
已经完成一次删列，当前已删除列数: 12, 当前BIC: -4007.45, 当前耗时 106 秒.
已经完成一次删列，当前已删除列数: 13, 当前BIC: -4014.49, 当前耗时 115 秒.
已经完成一次删列，当前已删除列数: 14, 当前BIC: -4021.53, 当前耗时 121 秒.
已经完成一次删列，当前已删除列数: 15, 当前BIC: -4028.57, 当前耗时 129 秒.
已经完成一次删列，当前已删除列数: 16, 当前BIC: -4035.6, 当前耗时 137 秒.
已经完成一次删列，当前已删除列数: 17, 当前BIC: -4042.62, 当前耗时 144 秒.
已经完成一次删列，当前已删除列数: 18, 当前BIC: -4049.65, 当前耗时 152 秒.
已经完成一次删列，当前已删除列数: 19, 当前BIC: -4056.68, 当前耗时 159 秒.
已经完成一次删列，当前已删除列数: 20, 当前BIC: -4063.69,

E:\Software\PythonProjects\Housing_Prices\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [8, 17, 19, 21] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


训练完成，花了54分钟。接下来看看这些参数

In [30]:
dict_scores = dict(scores.items())
print("fit_time: ", dict_scores["fit_time"].astype(int))
print("score_time: ", dict_scores["score_time"].round(4))
print("test_RMSE: ", -dict_scores["test_RMSE"].round(4))
print("train_RMSE: ", -dict_scores["train_RMSE"].round(4))
print("test_MAE: ", -dict_scores["test_MAE"].round(4))
print("train_MAE: ", -dict_scores["train_MAE"].round(4))
print("test_R2: ", dict_scores["test_R2"].round(4))
print("train_R2: ", dict_scores["train_R2"].round(4))

fit_time:  [668 671 628 690 604]
score_time:  [0.0204 0.0161 0.0159 0.0168 0.0189]
test_RMSE:  [0.2087 0.2325 0.2115 0.1405 0.1109]
train_RMSE:  [0.1054 0.1007 0.1012 0.1041 0.1076]
test_MAE:  [0.0876 0.1003 0.097  0.0809 0.0813]
train_MAE:  [0.0755 0.0735 0.0719 0.0744 0.0752]
test_R2:  [0.706  0.6758 0.714  0.8733 0.927 ]
train_R2:  [0.9316 0.9355 0.9361 0.9324 0.9264]


进一步计算一下均值和方差

In [32]:
print("train_RMSE", -scores["train_RMSE"].mean().round(4),"±", scores["train_RMSE"].std().round(4))
print("test_RMSE", -scores["test_RMSE"].mean().round(4),"±", scores["test_RMSE"].std().round(4))
print("train_MAE", -scores["train_MAE"].mean().round(4),"±", scores["train_MAE"].std().round(4))
print("test_MAE", -scores["test_MAE"].mean().round(4),"±", scores["test_MAE"].std().round(4))
print("train_R2", scores["train_R2"].mean().round(4),"±", scores["train_R2"].std().round(4))
print("test_R2", scores["test_R2"].mean().round(4),"±", scores["test_R2"].std().round(4))

train_RMSE 0.1038 ± 0.0026
test_RMSE 0.1808 ± 0.0467
train_MAE 0.0741 ± 0.0013
test_MAE 0.0894 ± 0.008
train_R2 -0.9324 ± 0.0035
test_R2 -0.7792 ± 0.101


现在就可以来看一下结果。首先是计算时间，这个基本上没有太大的差距，相对平稳，意料之内。我们比较一下之前的SetA，也就是baseline那边，把他们整理到一起
|scorings| Set A(baseline)| Set B (only selection)|
|--------|----------------|------------------------|
train_RMSE| 0.09356 ± 0.00267|0.1038 ± 0.0026|
test_RMSE| 0.15402 ± 0.0394|0.1808 ± 0.0467|
train_MAE| 0.06565 ± 0.0013|0.0741 ± 0.0013|
test_MAE| 0.08997 ± 0.0065|0.0894 ± 0.008|
train_R2| 0.94506 ± 0.00306|0.9324 ± 0.0035|
test_R2| 0.84298 ± 0.08305|0.7792 ± 0.101|



首先是RMSE，均方根误差，这次Set B的均方根误差，在测试集上的表现似乎不但没有明显提升，反而还稍微下降了一点。测试集上的则表现下降更明显，本来均方根误差只有0.15左右，现在变成了0.18左右；

MAE平均绝对值误差的表现类似，没有明显下降，基本持平甚至稍微变差一点点

R2最明显，训练集上面的差别不太明显，而测试集上的表现下降明显。

这其实提醒我们一件事，那就是BIC不是用来让预测变得更好的，他只是精简模型的，尤其是BIC的惩罚相对来说并不小，删列删的太多，影响总的预测结果也很正常。这里面有趣的是，MAE没有太大变化，即使是测试集也没有太大变化，而RMSE则降低稍微明显一点。我们知道RMSE是平方，对于比较大的误差，可以剧烈的放大，这也说明，可能是大部分普通数据的预测效果依然差不多保持了，但是对于少数难以预测的数据，误差被放大了。

并且，至少我们在这里证明，经过BIC选择的列，可以依旧保持和原来差距不太大的表现，原来有80列，经过编码以后，有210列，而BIC删去了160列，只剩50列，还能基本保持预测效果，这就凸显了其对模型的精简效果十分明显。

SetB的交叉验证做完，就可以开始SetC的了，SetC的和SetB的一样。同理，首先我们需要把之前的手动特征工程函数改成兼容transformer的形式。也就是至少得有fit()和transform()，而这里由于是纯数值计算，所以fit()不需要学习任何参数，第一步依然会执行fit()，但是没有任何效果，第二步的transform替换成前面的手工特征工程就行了，然后返回处理过后的数据集就可以了

In [37]:
from sklearn.base import TransformerMixin
class FeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        # * 年份可以用差值来代替建造时间
        X_input = X.copy()
        X_input["GarageAge"] = 0
        mask = X_input["HasGarage"] == 1
        X_input.loc[mask, "GarageAge"] = (X_input.loc[mask, "YrSold"] - X_input.loc[mask, "GarageYrBlt"])
        X_input["HouseAge"] = X_input["YrSold"] - X_input["YearBuilt"]
        X_input["RemodAge"] = X_input["YrSold"] - X_input["YearRemodAdd"]

        # * 可以把所有的面积加起来计算总使用面积
        X_input["TotalUtilSF"] = X_input["1stFlrSF"] + X_input["2ndFlrSF"] + X_input["TotalBsmtSF"]

        # * 卫生间等效加总
        X_input["EqvlBath"] = X_input["BsmtFullBath"] + 0.5 * X_input["BsmtHalfBath"] + X_input["FullBath"] + 0.5 * X_input["HalfBath"]

        # * 比例系列
        X_input["LivingRatio"] = X_input["GrLivArea"] / X_input["LotArea"]
        X_input["BedroomRatio"] = X_input["Bedroom"] / X_input["TotalUtilSF"]

        # * 面积×质量
        X_input["OvrallMul"] = X_input["OverallQual"] * X_input["LotArea"]
        X_input["BsmtMul"] = X_input["BsmtQual"] * X_input["TotalBsmtSF"]
        X_input["KitMul"] = X_input["Kitchen"] * X_input["KitchenQual"] # 这个是数量 * 质量

        # * Porch 总面积
        X_input["TotalPorchSF"] = X_input["OpenPorchSF"] + X_input["EnclosedPorch"] + X_input["3SsnPorch"] + X_input["ScreenPorch"]

        # * 设施存在性
        # HasGarage已经有了
        X_input["HasPool"] = 1 * (X_input["PoolArea"] > 0)
        X_input["HasBsmt"] = 1 * (X_input["BsmtQual"] > 0)
        X_input["HasFireplace"] = 1 * (X_input["Fireplaces"] > 0)
        X_input["Has2ndfloor"] = 1 * (X_input["2ndFlrSF"] > 0)
        X_input["HasPorch"] = 1 * (X_input["TotalPorchSF"] > 0)
        X_input["HasFence"] = 1 * (X_input["Fence"] > 0)

        # * 总设施数量
        X_input["Totalutil"] = X_input["HasPool"] + X_input["HasGarage"] + X_input["HasBsmt"] + X_input["HasFireplace"] + X_input["Has2ndfloor"] + X_input["HasPorch"] + X_input["HasFence"]

        # * 根号项
        X_input["RootUtil"] = X_input["Totalutil"] ** 0.5
        X_input["RootLotArea"] = X_input["LotArea"] ** 0.5

        return X_input


下一步，同样直接进行交叉验证，步骤和前面SetB一模一样，唯独只是pipeline前面多了一行特征工程。不过由于多了特征工程，所以preprocessor之前就多了很多列，不能取固定的列了，而是需要取新的列，把换成make_column_selector自动获取的

In [38]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.compose import make_column_selector
# 定义数据集
# num_cols = X.select_dtypes("number").columns
# cat_cols = X.select_dtypes("str").columns

print("数据已经准备好.")
# 定义preprocessor
# 使用make_column_selector自动获取列
preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), make_column_selector(dtype_include="number")),
        ("cat", OneHotEncoder(handle_unknown="ignore",
                              drop="first",
                              sparse_output=False),
         make_column_selector(dtype_exclude="number")),
    ]
)
preprocessor.set_output(transform="pandas")
print("处理器已定义.")

# 把处理器和模型连接起来
# 数据管道由两方面构成，一个是数据处理器，一个是线性模型，SetB没有特征工程环节。
pipeline = Pipeline(
    steps=[
        ("feature Engineering", FeatureEngineering()),
        ("preprocessor", preprocessor),
        ("selector", BICSelector()),
        ("LRmodel", LinearRegression()),
    ]
)
print("数据管道与模型已组装.")


# 做cross validate
# 定义Fold
cv = KFold(n_splits=5, shuffle=True, random_state=42)
# 使用cross_validate计算
# 先定义scorings，是我们需要用到的评价指标
scorings={
        "RMSE": "neg_root_mean_squared_error",
        "MAE": "neg_mean_absolute_error",
        "MSE": "neg_mean_squared_error",
        "R2": "r2"
    }

scores = cross_validate(
    pipeline,
    X,
    np.log(y),
    cv=cv,
    scoring=scorings,
    return_train_score=True
)
print("交叉验证已完成.")

数据已经准备好.
处理器已定义.
数据管道与模型已组装.
进入循环，开始测试各列并计算分数
已经完成一次删列，当前已删除列数: 1, 当前BIC: -3866.68, 当前耗时 9 秒.
已经完成一次删列，当前已删除列数: 2, 当前BIC: -3873.74, 当前耗时 17 秒.
已经完成一次删列，当前已删除列数: 3, 当前BIC: -3880.79, 当前耗时 25 秒.
已经完成一次删列，当前已删除列数: 4, 当前BIC: -3887.85, 当前耗时 33 秒.
已经完成一次删列，当前已删除列数: 5, 当前BIC: -3894.91, 当前耗时 41 秒.
已经完成一次删列，当前已删除列数: 6, 当前BIC: -3901.96, 当前耗时 48 秒.
已经完成一次删列，当前已删除列数: 7, 当前BIC: -3909.02, 当前耗时 56 秒.
已经完成一次删列，当前已删除列数: 8, 当前BIC: -3916.08, 当前耗时 63 秒.
已经完成一次删列，当前已删除列数: 9, 当前BIC: -3923.13, 当前耗时 70 秒.
已经完成一次删列，当前已删除列数: 10, 当前BIC: -3930.19, 当前耗时 79 秒.
已经完成一次删列，当前已删除列数: 11, 当前BIC: -3937.24, 当前耗时 87 秒.
已经完成一次删列，当前已删除列数: 12, 当前BIC: -3944.3, 当前耗时 95 秒.
已经完成一次删列，当前已删除列数: 13, 当前BIC: -3951.36, 当前耗时 103 秒.
已经完成一次删列，当前已删除列数: 14, 当前BIC: -3958.41, 当前耗时 111 秒.
已经完成一次删列，当前已删除列数: 15, 当前BIC: -3965.47, 当前耗时 124 秒.
已经完成一次删列，当前已删除列数: 16, 当前BIC: -3972.52, 当前耗时 132 秒.
已经完成一次删列，当前已删除列数: 17, 当前BIC: -3979.57, 当前耗时 140 秒.
已经完成一次删列，当前已删除列数: 18, 当前BIC: -3986.63, 当前耗时 148 秒.
已经完成一次删列，当前已删除列数: 19, 当前BIC: -3993.68, 当前耗时 156 秒.
已经完成一次删列

E:\Software\PythonProjects\Housing_Prices\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [12, 13] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


进入循环，开始测试各列并计算分数
已经完成一次删列，当前已删除列数: 1, 当前BIC: -4058.05, 当前耗时 12 秒.
已经完成一次删列，当前已删除列数: 2, 当前BIC: -4065.11, 当前耗时 19 秒.
已经完成一次删列，当前已删除列数: 3, 当前BIC: -4072.16, 当前耗时 31 秒.
已经完成一次删列，当前已删除列数: 4, 当前BIC: -4079.22, 当前耗时 42 秒.
已经完成一次删列，当前已删除列数: 5, 当前BIC: -4086.28, 当前耗时 53 秒.
已经完成一次删列，当前已删除列数: 6, 当前BIC: -4093.34, 当前耗时 62 秒.
已经完成一次删列，当前已删除列数: 7, 当前BIC: -4100.39, 当前耗时 70 秒.
已经完成一次删列，当前已删除列数: 8, 当前BIC: -4107.45, 当前耗时 79 秒.
已经完成一次删列，当前已删除列数: 9, 当前BIC: -4114.51, 当前耗时 89 秒.
已经完成一次删列，当前已删除列数: 10, 当前BIC: -4121.56, 当前耗时 97 秒.
已经完成一次删列，当前已删除列数: 11, 当前BIC: -4128.62, 当前耗时 110 秒.
已经完成一次删列，当前已删除列数: 12, 当前BIC: -4135.68, 当前耗时 117 秒.
已经完成一次删列，当前已删除列数: 13, 当前BIC: -4142.73, 当前耗时 125 秒.
已经完成一次删列，当前已删除列数: 14, 当前BIC: -4149.79, 当前耗时 132 秒.
已经完成一次删列，当前已删除列数: 15, 当前BIC: -4156.85, 当前耗时 140 秒.
已经完成一次删列，当前已删除列数: 16, 当前BIC: -4163.9, 当前耗时 151 秒.
已经完成一次删列，当前已删除列数: 17, 当前BIC: -4170.96, 当前耗时 160 秒.
已经完成一次删列，当前已删除列数: 18, 当前BIC: -4178.01, 当前耗时 167 秒.
已经完成一次删列，当前已删除列数: 19, 当前BIC: -4185.06, 当前耗时 174 秒.
已经完成一次删列，当前已删除列数: 20, 当前BIC: -4192

E:\Software\PythonProjects\Housing_Prices\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [8, 12, 13, 16, 21] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


进入循环，开始测试各列并计算分数
已经完成一次删列，当前已删除列数: 1, 当前BIC: -3956.44, 当前耗时 8 秒.
已经完成一次删列，当前已删除列数: 2, 当前BIC: -3963.5, 当前耗时 21 秒.
已经完成一次删列，当前已删除列数: 3, 当前BIC: -3970.55, 当前耗时 29 秒.
已经完成一次删列，当前已删除列数: 4, 当前BIC: -3977.61, 当前耗时 74 秒.
已经完成一次删列，当前已删除列数: 5, 当前BIC: -3984.67, 当前耗时 88 秒.
已经完成一次删列，当前已删除列数: 6, 当前BIC: -3991.73, 当前耗时 99 秒.
已经完成一次删列，当前已删除列数: 7, 当前BIC: -3998.78, 当前耗时 126 秒.
已经完成一次删列，当前已删除列数: 8, 当前BIC: -4005.84, 当前耗时 135 秒.
已经完成一次删列，当前已删除列数: 9, 当前BIC: -4012.9, 当前耗时 146 秒.
已经完成一次删列，当前已删除列数: 10, 当前BIC: -4019.95, 当前耗时 156 秒.
已经完成一次删列，当前已删除列数: 11, 当前BIC: -4027.01, 当前耗时 164 秒.
已经完成一次删列，当前已删除列数: 12, 当前BIC: -4034.07, 当前耗时 172 秒.
已经完成一次删列，当前已删除列数: 13, 当前BIC: -4041.12, 当前耗时 180 秒.
已经完成一次删列，当前已删除列数: 14, 当前BIC: -4048.18, 当前耗时 189 秒.
已经完成一次删列，当前已删除列数: 15, 当前BIC: -4055.24, 当前耗时 197 秒.
已经完成一次删列，当前已删除列数: 16, 当前BIC: -4062.3, 当前耗时 205 秒.
已经完成一次删列，当前已删除列数: 17, 当前BIC: -4069.35, 当前耗时 213 秒.
已经完成一次删列，当前已删除列数: 18, 当前BIC: -4076.41, 当前耗时 220 秒.
已经完成一次删列，当前已删除列数: 19, 当前BIC: -4083.46, 当前耗时 228 秒.
已经完成一次删列，当前已删除列数: 20, 当前BIC: -409

E:\Software\PythonProjects\Housing_Prices\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [8, 12, 14] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


进入循环，开始测试各列并计算分数
已经完成一次删列，当前已删除列数: 1, 当前BIC: -3908.34, 当前耗时 9 秒.
已经完成一次删列，当前已删除列数: 2, 当前BIC: -3915.4, 当前耗时 17 秒.
已经完成一次删列，当前已删除列数: 3, 当前BIC: -3922.45, 当前耗时 25 秒.
已经完成一次删列，当前已删除列数: 4, 当前BIC: -3929.51, 当前耗时 33 秒.
已经完成一次删列，当前已删除列数: 5, 当前BIC: -3936.57, 当前耗时 41 秒.
已经完成一次删列，当前已删除列数: 6, 当前BIC: -3943.63, 当前耗时 48 秒.
已经完成一次删列，当前已删除列数: 7, 当前BIC: -3950.68, 当前耗时 56 秒.
已经完成一次删列，当前已删除列数: 8, 当前BIC: -3957.74, 当前耗时 63 秒.
已经完成一次删列，当前已删除列数: 9, 当前BIC: -3964.8, 当前耗时 70 秒.
已经完成一次删列，当前已删除列数: 10, 当前BIC: -3971.85, 当前耗时 79 秒.
已经完成一次删列，当前已删除列数: 11, 当前BIC: -3978.91, 当前耗时 87 秒.
已经完成一次删列，当前已删除列数: 12, 当前BIC: -3985.97, 当前耗时 95 秒.
已经完成一次删列，当前已删除列数: 13, 当前BIC: -3993.03, 当前耗时 103 秒.
已经完成一次删列，当前已删除列数: 14, 当前BIC: -4000.08, 当前耗时 111 秒.
已经完成一次删列，当前已删除列数: 15, 当前BIC: -4007.14, 当前耗时 119 秒.
已经完成一次删列，当前已删除列数: 16, 当前BIC: -4014.19, 当前耗时 127 秒.
已经完成一次删列，当前已删除列数: 17, 当前BIC: -4021.25, 当前耗时 134 秒.
已经完成一次删列，当前已删除列数: 18, 当前BIC: -4028.3, 当前耗时 142 秒.
已经完成一次删列，当前已删除列数: 19, 当前BIC: -4035.35, 当前耗时 149 秒.
已经完成一次删列，当前已删除列数: 20, 当前BIC: -4042.4, 当

E:\Software\PythonProjects\Housing_Prices\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [13, 14] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


进入循环，开始测试各列并计算分数
已经完成一次删列，当前已删除列数: 1, 当前BIC: -3852.43, 当前耗时 7 秒.
已经完成一次删列，当前已删除列数: 2, 当前BIC: -3859.49, 当前耗时 15 秒.
已经完成一次删列，当前已删除列数: 3, 当前BIC: -3866.54, 当前耗时 22 秒.
已经完成一次删列，当前已删除列数: 4, 当前BIC: -3873.6, 当前耗时 30 秒.
已经完成一次删列，当前已删除列数: 5, 当前BIC: -3880.66, 当前耗时 37 秒.
已经完成一次删列，当前已删除列数: 6, 当前BIC: -3887.72, 当前耗时 44 秒.
已经完成一次删列，当前已删除列数: 7, 当前BIC: -3894.77, 当前耗时 53 秒.
已经完成一次删列，当前已删除列数: 8, 当前BIC: -3901.83, 当前耗时 61 秒.
已经完成一次删列，当前已删除列数: 9, 当前BIC: -3908.89, 当前耗时 69 秒.
已经完成一次删列，当前已删除列数: 10, 当前BIC: -3915.94, 当前耗时 77 秒.
已经完成一次删列，当前已删除列数: 11, 当前BIC: -3923.0, 当前耗时 85 秒.
已经完成一次删列，当前已删除列数: 12, 当前BIC: -3930.06, 当前耗时 93 秒.
已经完成一次删列，当前已删除列数: 13, 当前BIC: -3937.11, 当前耗时 101 秒.
已经完成一次删列，当前已删除列数: 14, 当前BIC: -3944.17, 当前耗时 108 秒.
已经完成一次删列，当前已删除列数: 15, 当前BIC: -3951.22, 当前耗时 116 秒.
已经完成一次删列，当前已删除列数: 16, 当前BIC: -3958.27, 当前耗时 123 秒.
已经完成一次删列，当前已删除列数: 17, 当前BIC: -3965.32, 当前耗时 131 秒.
已经完成一次删列，当前已删除列数: 18, 当前BIC: -3972.37, 当前耗时 138 秒.
已经完成一次删列，当前已删除列数: 19, 当前BIC: -3979.42, 当前耗时 145 秒.
已经完成一次删列，当前已删除列数: 20, 当前BIC: -3986.46,

E:\Software\PythonProjects\Housing_Prices\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [8, 17, 19, 21] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


顺利完成，接下来看一下效果

In [39]:
dict_scores = dict(scores.items())
print("fit_time: ", dict_scores["fit_time"].astype(int))
print("score_time: ", dict_scores["score_time"].round(4))
print("test_RMSE: ", -dict_scores["test_RMSE"].round(4))
print("train_RMSE: ", -dict_scores["train_RMSE"].round(4))
print("test_MAE: ", -dict_scores["test_MAE"].round(4))
print("train_MAE: ", -dict_scores["train_MAE"].round(4))
print("test_R2: ", dict_scores["test_R2"].round(4))
print("train_R2: ", dict_scores["train_R2"].round(4))

fit_time:  [691 703 819 662 632]
score_time:  [0.0252 0.026  0.0378 0.0254 0.025 ]
test_RMSE:  [0.2013 0.2421 0.1878 0.1442 0.1215]
train_RMSE:  [0.1041 0.0996 0.1002 0.1036 0.105 ]
test_MAE:  [0.0878 0.1028 0.0926 0.0804 0.0839]
train_MAE:  [0.0746 0.0712 0.0705 0.0726 0.073 ]
test_R2:  [0.7263 0.6486 0.7747 0.8666 0.9123]
train_R2:  [0.9332 0.9369 0.9374 0.9329 0.9299]


In [40]:
print("train_RMSE", -scores["train_RMSE"].mean().round(4),"±", scores["train_RMSE"].std().round(4))
print("test_RMSE", -scores["test_RMSE"].mean().round(4),"±", scores["test_RMSE"].std().round(4))
print("train_MAE", -scores["train_MAE"].mean().round(4),"±", scores["train_MAE"].std().round(4))
print("test_MAE", -scores["test_MAE"].mean().round(4),"±", scores["test_MAE"].std().round(4))
print("train_R2", scores["train_R2"].mean().round(4),"±", scores["train_R2"].std().round(4))
print("test_R2", scores["test_R2"].mean().round(4),"±", scores["test_R2"].std().round(4))

train_RMSE 0.1025 ± 0.0022
test_RMSE 0.1794 ± 0.0426
train_MAE 0.0724 ± 0.0015
test_MAE 0.0895 ± 0.0078
train_R2 0.9341 ± 0.0028
test_R2 0.7857 ± 0.0949


再对比一下：

|scorings| Set A(baseline)| Set B (only selection)| Set C (FN+selection)|
|--------|----------------|------------------------|------|
train_RMSE| 0.09356 ± 0.00267|0.1038 ± 0.0026|0.1025 ± 0.0022|
test_RMSE| 0.15402 ± 0.0394|0.1808 ± 0.0467|0.1794 ± 0.0426|
train_MAE| 0.06565 ± 0.0013|0.0741 ± 0.0013|0.0724 ± 0.0015|
test_MAE| 0.08997 ± 0.0065|0.0894 ± 0.008|0.0895 ± 0.0078|
train_R2| 0.94506 ± 0.00306|0.9324 ± 0.0035|0.9341 ± 0.0028|
test_R2| 0.84298 ± 0.08305|0.7792 ± 0.101|0.7857 ± 0.0949|

如果仔细观察，就发现set C其实对比set B还是有轻微的提升的，RMSE稍微降低了，MAE也稍微降低了，R2也都稍微增加了，不过显然，手工特征工程似乎并不能抵挡BIC的损害。更细节的分析我们不如放到云电脑里面跑一次完整的演示，然后再分析吧。

这里我们先提交Kaggle项目吧，也不管效果了，直接用baseline预测吧，反正就他表现最好。而且很快，不用在等一个小时了。我们再一次使用pipeline，使用全部数据，不split

In [42]:
# 数据

num_cols = X.select_dtypes("number").columns
cat_cols = X.select_dtypes("str").columns

# 处理器定义
preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore",
                              drop="first",
                              sparse_output=False),
         cat_cols),
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("LRmodel", LinearRegression()),
    ]
)

# 测试pipeline
# 数据应该是刚刚从df里面分出来的X和y，y提前取log就行了，其他的不要做，不能重复处理
pipeline.fit(X, np.log(y))
print("基线模型训练完成.") # 加一个防止输出html进而导致渲染错误

基线模型训练完成.


fit结束，现在开始预测，终于可以把测试集数据拿出来了。要注意两点，第一